# PSE Moderator Comparison + Causal Mediation with Quadrant Diagnostics

Full econometric implementation of **PSE (ILO public-sector employment share)** as a moderator
and causal mediator in the **Education × Gini × SocProt → Democracy** triple-interaction
framework, on a panel of democratizing countries (periods 2015–19 and 2020–22).

**Key analyses:**
1. **DD triple interaction** — SocProt and PSE as alternating moderators (Giesselmann–Schmidt-Catran within-DD estimator with HC3 robust SEs)
2. **Baron-Kenny causal mediation** — bootstrapped ACME via PSE → judicial compliance / NGO / liberal democracy
3. **Quadrant missing-data diagnostics** — PSE coverage by Gini × SocProt quadrant
4. **Sensitivity checks** — extreme-value ACME bounds, sequential ignorability (ρ), lagged SocProt
5. **Reverse causality tests** — lagged LDem→Gini, IMF SAP→ΔSocProt

**Data:** V-Dem v15 × ILO SDG 1.3.1 × SWIID × ILO ILOSTAT PSE × Dreher (2006) IMF dummies
(demo uses a curated 36-observation subset covering 18 democratizing countries)

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT pre-installed on Colab, always install
_pip('loguru==0.7.3')

# Core scientific packages — pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3', 'statsmodels==0.14.6', 'matplotlib==3.10.0')

In [ ]:
import gc
import json
import warnings
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from loguru import logger

warnings.filterwarnings("ignore", category=RuntimeWarning)

logger.remove()
logger.add(lambda msg: print(msg, end=""), level="INFO", colorize=False,
           format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-395f4e-education-inequality-and-democratic-eros/main/round-2/experiment-2/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    import os
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
raw = load_data()
panel = pd.DataFrame(raw["panel"])
logger.info(f"Loaded panel: {len(panel)} rows, {panel['country_iso3'].nunique()} countries")
panel.head()

In [ ]:
N_BOOTSTRAP = 200   # Baron-Kenny bootstrap replicates (original: 1000)
SEED = 42

## Helper: safe_float

Utility used throughout: converts any value to float, returning `None` for NaN, None, or non-finite results. Prevents silent NA propagation in coefficient extraction.

In [ ]:
def safe_float(x: Any) -> float | None:
    """Convert value to float, returning None for NaN/None/non-finite."""
    if x is None:
        return None
    try:
        v = float(x)
        return None if not np.isfinite(v) else v
    except (TypeError, ValueError):
        return None

## Section 0 — Build Analysis Subsamples

From the loaded panel, construct three subsets used in different analyses:
- **`complete`** — rows where `has_all_core=True` (used for DD SocProt and reverse causality)
- **`pse_sample`** — rows with PSE data (for DD PSE moderator spec; cross-sectional fallback)
- **`med_sample`** — rows with PSE + judicial compliance + liberal democracy (for Baron-Kenny mediation)

In [ ]:
complete = panel[panel["has_all_core"] == True].copy()
logger.info(f"Complete panel (has_all_core=True): {len(complete)} rows, {complete['country_iso3'].nunique()} countries")

# PSE moderator sample: education + gini + PSE all non-null (no SocProt required)
pse_sample = panel.dropna(subset=["education", "gini_disp", "pse_share_period", "v2x_libdem"]).copy()
# Recompute within-country deviations on the PSE sample (different composition → different means)
pse_sample["mean_edu_pse"]  = pse_sample.groupby("country_iso3")["education"].transform("mean")
pse_sample["mean_gini_pse"] = pse_sample.groupby("country_iso3")["gini_disp"].transform("mean")
pse_sample["e_edu_pse"]     = pse_sample["education"] - pse_sample["mean_edu_pse"]
pse_sample["e_gini_pse"]    = pse_sample["gini_disp"] - pse_sample["mean_gini_pse"]
logger.info(f"PSE moderator sample: {len(pse_sample)} rows, {pse_sample['country_iso3'].nunique()} countries")

# Mediation sample: education + gini + socprot + PSE + v2jucomp all non-null
med_sample = panel.dropna(
    subset=["education", "gini_disp", "socprot_coverage", "pse_share_period",
            "v2jucomp", "v2x_libdem"]
).copy()
logger.info(f"Mediation sample: {len(med_sample)} rows, {med_sample['country_iso3'].nunique()} countries")

## Section 1 — Quadrant Missing-Data Diagnostic

PSE coverage broken out by Gini × SocProt quadrant. Reveals whether PSE data gaps are systematic (e.g., concentrated in high-inequality / low-SocProt quadrant) or random.

In [ ]:
def compute_quadrant_coverage_table(df: pd.DataFrame) -> dict:
    sub = df.dropna(subset=["gini_disp", "socprot_coverage"]).copy()
    gini_med = sub["gini_disp"].median()
    socprot_med = sub["socprot_coverage"].median()
    sub["quad_highgini"] = (sub["gini_disp"] > gini_med).astype(int)
    sub["quad_highsocprot"] = (sub["socprot_coverage"] > socprot_med).astype(int)

    table: dict = {}
    for hg in [0, 1]:
        for hs in [0, 1]:
            cell = sub[(sub["quad_highgini"] == hg) & (sub["quad_highsocprot"] == hs)]
            n_total = len(cell)
            n_pse = int(cell["pse_share_period"].notna().sum())
            label = f"highGini={hg}_highSocProt={hs}"
            table[label] = {
                "n_observations": n_total,
                "n_with_pse": n_pse,
                "pse_coverage_pct": round(100 * n_pse / n_total, 1) if n_total > 0 else 0.0,
                "countries": sorted(cell["country_iso3"].unique().tolist()),
            }
    logger.info(f"Quadrant table computed; gini_med={gini_med:.2f}, socprot_med={socprot_med:.2f}")
    for k, v in table.items():
        logger.info(f"  {k}: n={v['n_observations']}, pse_coverage={v['pse_coverage_pct']}%")
    return table


pse_missing_table = compute_quadrant_coverage_table(complete)
pse_missing_table

## Section 2 — DD Triple Interaction Estimator

Giesselmann–Schmidt-Catran within-country DD triple interaction: **Education × Gini × Moderator → Liberal Democracy**.

- **SocProt spec**: moderator = social protection coverage (reference model, within-DD feasible)
- **PSE spec**: moderator = public-sector employment share (cross-sectional fallback; all PSE obs are 2015 singletons)

HC3 robust SEs throughout. β₇ is the triple interaction coefficient.

In [ ]:
def run_dd_triple(
    df: pd.DataFrame,
    e_educ_col: str,
    e_gini_col: str,
    e_mod_col: str,
    outcome_col: str,
    country_col: str,
    label: str,
) -> dict:
    required = [e_educ_col, e_gini_col, e_mod_col, outcome_col, country_col, "period_start"]
    df = df.dropna(subset=required).copy()
    n_obs_raw = len(df)
    n_ctry = df[country_col].nunique()

    if n_obs_raw < 10:
        logger.warning(f"{label}: insufficient obs ({n_obs_raw})")
        return {"error": f"Insufficient obs ({n_obs_raw}) for {label} spec", "n_obs": n_obs_raw}

    obs_per_country = df.groupby(country_col).size()
    multi_period_countries = obs_per_country[obs_per_country >= 2].index
    n_multi = len(multi_period_countries)
    use_dd = n_multi >= 5

    if not use_dd:
        logger.warning(
            f"{label}: only {n_multi} countries have 2+ periods — "
            f"within-country DD not feasible; using cross-sectional OLS fallback"
        )
        return _run_cross_sectional_fallback(df, e_educ_col, e_gini_col, e_mod_col, outcome_col, label)

    df["int_eg"]  = df[e_educ_col] * df[e_gini_col]
    df["int_em"]  = df[e_educ_col] * df[e_mod_col]
    df["int_gm"]  = df[e_gini_col] * df[e_mod_col]
    df["int_egm"] = df[e_educ_col] * df[e_gini_col] * df[e_mod_col]

    for col in ["int_eg", "int_em", "int_gm", "int_egm"]:
        mean_col = df.groupby(country_col)[col].transform("mean")
        df[f"dd_{col}"] = df[col] - mean_col

    df["y_within"] = df[outcome_col] - df.groupby(country_col)[outcome_col].transform("mean")

    period_dummies = pd.get_dummies(df["period_start"], prefix="pd", drop_first=True).astype(float)
    x_cols = [e_educ_col, e_gini_col, e_mod_col,
               "dd_int_eg", "dd_int_em", "dd_int_gm", "dd_int_egm"]
    X = pd.concat([df[x_cols].reset_index(drop=True), period_dummies.reset_index(drop=True)], axis=1)
    X = add_constant(X)
    y = df["y_within"].reset_index(drop=True)

    for drop_col in ["dd_int_gm", "dd_int_em"]:
        rank = np.linalg.matrix_rank(X.values.astype(float))
        if rank < X.shape[1]:
            if drop_col in X.columns:
                logger.warning(f"{label}: rank deficiency (rank={rank}, ncols={X.shape[1]}), dropping {drop_col}")
                X = X.drop(columns=[drop_col])

    model = OLS(y, X, missing="drop").fit(cov_type="HC3")

    beta7 = safe_float(model.params.get("dd_int_egm"))
    se7   = safe_float(model.bse.get("dd_int_egm"))
    t7    = safe_float(model.tvalues.get("dd_int_egm"))
    p7    = safe_float(model.pvalues.get("dd_int_egm"))

    p25m = float(df[e_mod_col].quantile(0.25))
    p75m = float(df[e_mod_col].quantile(0.75))
    p75g = float(df[e_gini_col].quantile(0.75))
    b1 = safe_float(model.params.get(e_educ_col)) or 0.0
    b4 = safe_float(model.params.get("dd_int_eg"))  or 0.0
    b5 = safe_float(model.params.get("dd_int_em"))  or 0.0
    b7 = beta7 or 0.0
    me_low_m  = b1 + b4 * p75g + b5 * p25m + b7 * p75g * p25m
    me_high_m = b1 + b4 * p75g + b5 * p75m + b7 * p75g * p75m

    def _safe_coef(k):
        c = safe_float(model.params.get(k))
        p = safe_float(model.pvalues.get(k))
        if c is None:
            return None
        return {"coef": round(c, 6), "p": round(p, 4) if p is not None else None}

    all_coefs = {k: _safe_coef(k) for k in model.params.index if _safe_coef(k) is not None}

    country_means_s = df.groupby(country_col)[outcome_col].mean()
    country_mean_arr = df[country_col].map(country_means_s).values
    fv_raw = model.fittedvalues.values
    fitted_level = fv_raw + country_mean_arr[:len(fv_raw)]
    fitted_values = {
        int(orig_idx): round(float(fv), 6) if np.isfinite(fv) else None
        for orig_idx, fv in zip(df.index[:len(fv_raw)], fitted_level)
    }

    result = {
        "moderator": label,
        "estimation": "within_DD",
        "n_obs": int(model.nobs),
        "n_countries": n_ctry,
        "n_multi_period_countries": n_multi,
        "beta7": round(beta7, 6) if beta7 is not None else None,
        "se7":   round(se7,   6) if se7   is not None else None,
        "t7":    round(t7,    4) if t7    is not None else None,
        "p7":    round(p7,    4) if p7    is not None else None,
        "sign_reversal": {
            "me_at_low_moderator":  round(me_low_m,  6),
            "me_at_high_moderator": round(me_high_m, 6),
            "signs_differ": bool(me_low_m * me_high_m < 0),
        },
        "rsquared": safe_float(model.rsquared),
        "all_coefs": all_coefs,
        "_fitted_values": fitted_values,
    }
    logger.info(
        f"{label}: n={result['n_obs']}, β₇={result['beta7']}, p={result['p7']}, "
        f"sign_reversal={result['sign_reversal']['signs_differ']}"
    )
    return result


def _run_cross_sectional_fallback(
    df: pd.DataFrame,
    e_educ_col: str,
    e_gini_col: str,
    e_mod_col: str,
    outcome_col: str,
    label: str,
) -> dict:
    period_dummies = pd.get_dummies(df["period_start"], prefix="pd", drop_first=True).astype(float)
    df = df.copy()
    df["int_eg"]  = df[e_educ_col] * df[e_gini_col]
    df["int_em"]  = df[e_educ_col] * df[e_mod_col]
    df["int_gm"]  = df[e_gini_col] * df[e_mod_col]
    df["int_egm"] = df[e_educ_col] * df[e_gini_col] * df[e_mod_col]
    x_cols = [e_educ_col, e_gini_col, e_mod_col, "int_eg", "int_em", "int_gm", "int_egm"]
    X = pd.concat([df[x_cols].reset_index(drop=True), period_dummies.reset_index(drop=True)], axis=1)
    X = add_constant(X)
    y = df[outcome_col].reset_index(drop=True)

    for drop_col in ["int_gm", "int_em"]:
        rank = np.linalg.matrix_rank(X.values.astype(float))
        if rank < X.shape[1] and drop_col in X.columns:
            logger.warning(f"{label} (XS): dropping {drop_col} due to rank deficiency")
            X = X.drop(columns=[drop_col])

    try:
        model = OLS(y, X, missing="drop").fit(cov_type="HC3")
    except Exception as exc:
        logger.error(f"{label} cross-sectional OLS failed: {exc}")
        return {"error": str(exc), "n_obs": len(df), "estimation": "cross_sectional_failed"}

    beta7 = safe_float(model.params.get("int_egm"))
    se7   = safe_float(model.bse.get("int_egm"))
    t7    = safe_float(model.tvalues.get("int_egm"))
    p7    = safe_float(model.pvalues.get("int_egm"))

    p25m = float(df[e_mod_col].quantile(0.25))
    p75m = float(df[e_mod_col].quantile(0.75))
    p75g = float(df[e_gini_col].quantile(0.75))
    b1   = safe_float(model.params.get(e_educ_col))  or 0.0
    b4   = safe_float(model.params.get("int_eg"))    or 0.0
    b5   = safe_float(model.params.get("int_em"))    or 0.0
    b7   = beta7 or 0.0
    me_low_m  = b1 + b4 * p75g + b5 * p25m + b7 * p75g * p25m
    me_high_m = b1 + b4 * p75g + b5 * p75m + b7 * p75g * p75m

    def _safe_coef(k):
        c = safe_float(model.params.get(k))
        p = safe_float(model.pvalues.get(k))
        if c is None:
            return None
        return {"coef": round(c, 6), "p": round(p, 4) if p is not None else None}

    all_coefs = {k: _safe_coef(k) for k in model.params.index if _safe_coef(k) is not None}

    fv_raw = model.fittedvalues.values
    fitted_values = {
        int(orig_idx): round(float(fv), 6) if np.isfinite(fv) else None
        for orig_idx, fv in zip(df.index[:len(fv_raw)], fv_raw)
    }

    result = {
        "moderator": label,
        "estimation": "cross_sectional_OLS_no_country_FE",
        "caveat": (
            "DD within-country estimator not feasible (all countries are singletons in PSE data). "
            "Cross-sectional OLS without country fixed effects. "
            "Coefficients reflect between-country variation and may be confounded."
        ),
        "n_obs": int(model.nobs),
        "n_countries": df["country_iso3"].nunique() if "country_iso3" in df.columns else None,
        "n_multi_period_countries": 0,
        "beta7": round(beta7, 6) if beta7 is not None else None,
        "se7":   round(se7,   6) if se7   is not None else None,
        "t7":    round(t7,    4) if t7    is not None else None,
        "p7":    round(p7,    4) if p7    is not None else None,
        "sign_reversal": {
            "me_at_low_moderator":  round(me_low_m,  6),
            "me_at_high_moderator": round(me_high_m, 6),
            "signs_differ": bool(me_low_m * me_high_m < 0),
        },
        "rsquared": safe_float(model.rsquared),
        "all_coefs": all_coefs,
        "_fitted_values": fitted_values,
    }
    logger.info(
        f"{label} (XS): n={result['n_obs']}, β₇={result['beta7']}, p={result['p7']}, "
        f"sign_reversal={result['sign_reversal']['signs_differ']}"
    )
    return result


# ── Run DD triple: SocProt moderator (reference) ──────────────────────────
logger.info("--- DD triple interaction: SocProt moderator (reference) ---")
result_socprot = run_dd_triple(
    df=complete,
    e_educ_col="e_education",
    e_gini_col="e_gini_disp",
    e_mod_col="e_socprot_coverage",
    outcome_col="v2x_libdem",
    country_col="country_iso3",
    label="SocProt_coverage",
)
print(f"\nSocProt β₇={result_socprot.get('beta7')}, p={result_socprot.get('p7')}, "
      f"n={result_socprot.get('n_obs')}, estimation={result_socprot.get('estimation')}")

# ── Run DD triple: PSE moderator (cross-sectional fallback expected) ──────
logger.info("--- DD triple interaction: PSE moderator ---")
result_pse = run_dd_triple(
    df=pse_sample,
    e_educ_col="education",
    e_gini_col="gini_disp",
    e_mod_col="pse_share_period",
    outcome_col="v2x_libdem",
    country_col="country_iso3",
    label="PSE_share",
)
print(f"\nPSE β₇={result_pse.get('beta7')}, p={result_pse.get('p7')}, "
      f"n={result_pse.get('n_obs')}, estimation={result_pse.get('estimation')}")

## Section 3 — Baron-Kenny Causal Mediation

Bootstrapped ACME for the path: **Edu×Gini×(−SocProt) → PSE share → {judicial compliance, NGO activity, liberal democracy}**.

All variables within-country demeaned; period FEs included. With N=15 PSE singletons (all 2015), within-demeaning collapses T→0 — cross-sectional fallback is used automatically.

In [ ]:
def run_baron_kenny_mediation(
    df: pd.DataFrame,
    outcome_col: str,
    mediator_col: str,
    n_bootstrap: int = 1000,
    seed: int = 42,
) -> dict:
    required = ["e_education", "e_gini_disp", "e_socprot_coverage", mediator_col, outcome_col,
                "country_iso3", "period_start"]
    df = df.dropna(subset=required).copy()
    n = len(df)

    if n < 15:
        return {"error": "Insufficient obs for mediation", "insufficient_data": True, "n_obs": n}

    df["T_raw"] = df["e_education"] * df["e_gini_disp"] * (-df["e_socprot_coverage"])
    df["T"] = df["T_raw"] - df.groupby("country_iso3")["T_raw"].transform("mean")

    cross_sectional_fallback = df["T"].abs().max() < 1e-10
    if cross_sectional_fallback:
        logger.warning(
            f"Mediation ({outcome_col}): T within-demeaned to 0 (singleton-period sample). "
            f"Using level triple product education×gini×(−socprot) as cross-sectional exposure."
        )
        df["T"] = df["education"] * df["gini_disp"] * (-df["socprot_coverage"])
        t_std = df["T"].std()
        if t_std > 1e-10:
            df["T"] = df["T"] / t_std

    df["M"] = df[mediator_col] - df.groupby("country_iso3")[mediator_col].transform("mean")
    df["Y"] = df[outcome_col]  - df.groupby("country_iso3")[outcome_col].transform("mean")

    if cross_sectional_fallback:
        df["M"] = df[mediator_col]
        df["Y"] = df[outcome_col]

    period_dummies = pd.get_dummies(df["period_start"], prefix="pd", drop_first=True).astype(float)

    def fit_ols(y_data, regressors):
        X = pd.concat(
            [regressors.reset_index(drop=True), period_dummies.reset_index(drop=True)], axis=1
        )
        X = add_constant(X)
        y = y_data.reset_index(drop=True)
        valid = y.notna() & X.notna().all(axis=1)
        return OLS(y[valid], X[valid], missing="drop").fit(cov_type="HC3")

    T_df = df[["T"]].reset_index(drop=True)
    M_df = df[["M"]].reset_index(drop=True)

    mod_c  = fit_ols(df["Y"], T_df)
    c_coef = safe_float(mod_c.params.get("T"))
    c_p    = safe_float(mod_c.pvalues.get("T"))

    mod_a  = fit_ols(df["M"], T_df)
    a_coef = safe_float(mod_a.params.get("T"))
    a_p    = safe_float(mod_a.pvalues.get("T"))

    TM_df  = pd.concat([T_df, M_df], axis=1)
    mod_c2 = fit_ols(df["Y"], TM_df)
    b_coef  = safe_float(mod_c2.params.get("M"))
    b_p     = safe_float(mod_c2.pvalues.get("M"))
    cp_coef = safe_float(mod_c2.params.get("T"))
    cp_p    = safe_float(mod_c2.pvalues.get("T"))

    acme = (a_coef * b_coef) if (a_coef is not None and b_coef is not None) else None
    prop = (acme / c_coef) if (c_coef is not None and abs(c_coef) > 1e-12 and acme is not None) else None

    rng = np.random.default_rng(seed)
    acme_boot: list[float] = []
    idx_all = np.arange(n)
    pd_arr  = period_dummies.values

    for _ in range(n_bootstrap):
        bi = rng.choice(idx_all, size=n, replace=True)
        T_b  = df["T"].values[bi].reshape(-1, 1)
        M_b  = df["M"].values[bi].reshape(-1, 1)
        Y_b  = df["Y"].values[bi]
        pd_b = pd_arr[bi]
        try:
            Xa = np.column_stack([np.ones(n), T_b, pd_b])
            Xc = np.column_stack([np.ones(n), T_b, M_b, pd_b])
            valid_a = np.isfinite(Xa).all(axis=1) & np.isfinite(M_b.ravel())
            valid_c = np.isfinite(Xc).all(axis=1) & np.isfinite(Y_b)
            if valid_a.sum() < 5 or valid_c.sum() < 5:
                continue
            a_b = np.linalg.lstsq(Xa[valid_a], M_b.ravel()[valid_a], rcond=None)[0][1]
            b_b = np.linalg.lstsq(Xc[valid_c], Y_b[valid_c], rcond=None)[0][2]
            acme_boot.append(float(a_b * b_b))
        except np.linalg.LinAlgError:
            pass

    n_valid_boot = len(acme_boot)
    acme_ci_low  = float(np.percentile(acme_boot, 2.5))  if acme_boot else None
    acme_ci_high = float(np.percentile(acme_boot, 97.5)) if acme_boot else None
    bootstrap_unstable = n_valid_boot < 50

    result = {
        "outcome": outcome_col,
        "mediator": mediator_col,
        "n_obs": n,
        "n_countries": int(df["country_iso3"].nunique()),
        "estimation": "cross_sectional_OLS_no_country_FE" if cross_sectional_fallback else "within_country_FE",
        "caveat": (
            "All PSE obs are period_start=2015; within-country demeaning collapses T→0. "
            "Cross-sectional mediation using level education×gini×(−socprot) as exposure."
        ) if cross_sectional_fallback else None,
        "total_effect_c":     round(c_coef,  6) if c_coef  is not None else None,
        "total_effect_c_p":   round(c_p,     4) if c_p     is not None else None,
        "a_coef_T_to_M":      round(a_coef,  6) if a_coef  is not None else None,
        "a_p":                round(a_p,     4) if a_p     is not None else None,
        "b_coef_M_to_Y":      round(b_coef,  6) if b_coef  is not None else None,
        "b_p":                round(b_p,     4) if b_p     is not None else None,
        "direct_effect_cp":   round(cp_coef, 6) if cp_coef is not None else None,
        "direct_effect_cp_p": round(cp_p,    4) if cp_p    is not None else None,
        "acme": round(acme, 6) if acme is not None else None,
        "acme_ci_95": (
            [round(acme_ci_low, 6), round(acme_ci_high, 6)]
            if acme_ci_low is not None else None
        ),
        "n_bootstrap_valid": n_valid_boot,
        "bootstrap_unstable": bootstrap_unstable,
        "proportion_mediated": round(prop, 4) if prop is not None else None,
        "baron_kenny_satisfied": bool(
            (c_p is not None and c_p < 0.05) and
            (a_p is not None and a_p < 0.05) and
            (b_p is not None and b_p < 0.05)
        ),
    }
    logger.info(
        f"Mediation ({outcome_col}): ACME={result['acme']}, a_p={result['a_p']}, "
        f"b_p={result['b_p']}, BK_satisfied={result['baron_kenny_satisfied']}"
    )
    return result


if len(med_sample) >= 15:
    med_judicial = run_baron_kenny_mediation(
        med_sample, outcome_col="v2jucomp", mediator_col="pse_share_period",
        n_bootstrap=N_BOOTSTRAP, seed=SEED,
    )
    med_ngo = run_baron_kenny_mediation(
        med_sample, outcome_col="v2cseeorgs", mediator_col="pse_share_period",
        n_bootstrap=N_BOOTSTRAP, seed=SEED,
    )
    med_libdem = run_baron_kenny_mediation(
        med_sample, outcome_col="v2x_libdem", mediator_col="pse_share_period",
        n_bootstrap=N_BOOTSTRAP, seed=SEED,
    )
else:
    logger.warning(f"Mediation sample too small ({len(med_sample)})")
    _empty = {"error": "mediation_sample_too_small", "n_obs": len(med_sample), "insufficient_data": True}
    med_judicial = med_ngo = med_libdem = _empty

print(f"Judicial ACME={med_judicial.get('acme')}, CI={med_judicial.get('acme_ci_95')}, BK={med_judicial.get('baron_kenny_satisfied')}")
print(f"NGO     ACME={med_ngo.get('acme')}, CI={med_ngo.get('acme_ci_95')}, BK={med_ngo.get('baron_kenny_satisfied')}")
print(f"LDem    ACME={med_libdem.get('acme')}, CI={med_libdem.get('acme_ci_95')}, BK={med_libdem.get('baron_kenny_satisfied')}")

## Section 4 — Sensitivity Checks

- **Extreme-values ACME bound**: assumes ACME=0 for countries without PSE data; computes the attenuated ACME if zeros are correct
- **Sequential ignorability check**: ρ = residual correlation between mediator and outcome equations; large |ρ| with p<0.05 indicates potential unobserved confounding

In [ ]:
def compute_extreme_sensitivity(
    complete: pd.DataFrame,
    med_sample: pd.DataFrame,
    acme_observed: float | None,
) -> dict:
    missing_pse = complete[complete["pse_share_period"].isna()]["country_iso3"].unique().tolist()
    n_missing   = len(missing_pse)
    n_with_pse  = int(med_sample["country_iso3"].nunique())
    n_total     = int(complete["country_iso3"].nunique())

    if acme_observed is None or abs(acme_observed) < 1e-15:
        return {
            "n_countries_with_pse_data": n_with_pse,
            "n_countries_total_complete": n_total,
            "n_countries_missing_pse": n_missing,
            "missing_pse_country_list": sorted(missing_pse),
            "acme_observed": None,
            "acme_weighted_conservative_assuming_zeros": None,
            "pct_attenuation": None,
            "survives_conservative_bound": False,
        }

    acme_weighted = float(acme_observed) * n_with_pse / n_total
    return {
        "n_countries_with_pse_data": n_with_pse,
        "n_countries_total_complete": n_total,
        "n_countries_missing_pse": n_missing,
        "missing_pse_country_list": sorted(missing_pse),
        "acme_observed": round(float(acme_observed), 6),
        "acme_weighted_conservative_assuming_zeros": round(acme_weighted, 6),
        "pct_attenuation": round(100 * (1 - acme_weighted / float(acme_observed)), 1),
        "survives_conservative_bound": bool(abs(acme_weighted) > 0.001),
    }


def sequential_ignorability_check(
    df: pd.DataFrame,
    outcome_col: str = "v2jucomp",
    mediator_col: str = "pse_share_period",
) -> dict:
    required = ["e_education", "e_gini_disp", "e_socprot_coverage",
                mediator_col, outcome_col, "country_iso3", "period_start"]
    df = df.dropna(subset=required).copy()
    if len(df) < 10:
        return {"error": "Insufficient obs for sequential ignorability check", "n_obs": len(df)}

    df["T_raw"] = df["e_education"] * df["e_gini_disp"] * (-df["e_socprot_coverage"])
    df["T"] = df["T_raw"] - df.groupby("country_iso3")["T_raw"].transform("mean")

    cross_sectional = df["T"].abs().max() < 1e-10
    if cross_sectional:
        df["T"] = df["education"] * df["gini_disp"] * (-df["socprot_coverage"])
        t_std = df["T"].std()
        if t_std > 1e-10:
            df["T"] = df["T"] / t_std
        df["M"] = df[mediator_col]
        df["Y"] = df[outcome_col]
    else:
        df["M"] = df[mediator_col] - df.groupby("country_iso3")[mediator_col].transform("mean")
        df["Y"] = df[outcome_col]  - df.groupby("country_iso3")[outcome_col].transform("mean")

    period_dummies = pd.get_dummies(df["period_start"], prefix="pd", drop_first=True).astype(float)
    T_df = df[["T"]].reset_index(drop=True)
    pd_r = period_dummies.reset_index(drop=True)

    Xa = add_constant(pd.concat([T_df, pd_r], axis=1))
    mod_a = OLS(df["M"].reset_index(drop=True), Xa, missing="drop").fit()
    resid_m = mod_a.resid

    Xc = add_constant(pd.concat([T_df, pd_r], axis=1))
    mod_c = OLS(df["Y"].reset_index(drop=True), Xc, missing="drop").fit()
    resid_y = mod_c.resid

    min_len = min(len(resid_m), len(resid_y))
    rm = resid_m.values[:min_len]
    ry = resid_y.values[:min_len]
    rho_val = rho_p_val = None
    if np.std(rm) > 1e-12 and np.std(ry) > 1e-12:
        rho_raw, rho_p_raw = stats.pearsonr(rm, ry)
        rho_val   = safe_float(rho_raw)
        rho_p_val = safe_float(rho_p_raw)

    result = {
        "rho_residual_correlation": round(rho_val, 4) if rho_val is not None else None,
        "rho_p_value": round(rho_p_val, 4) if rho_p_val is not None else None,
        "sequential_ignorability_concern": bool(
            rho_val is not None and rho_p_val is not None and abs(rho_val) > 0.3 and rho_p_val < 0.05
        ),
        "interpretation": (
            "Non-zero rho suggests unobserved confounders affect both PSE selection and outcome; "
            "this would violate sequential ignorability and bias ACME estimates."
        ),
    }
    logger.info(
        f"Sequential ignorability: rho={result['rho_residual_correlation']}, "
        f"concern={result['sequential_ignorability_concern']}"
    )
    return result


# ── Extreme-values ACME sensitivity ──────────────────────────────────────
acme_obs = med_judicial.get("acme") if isinstance(med_judicial, dict) else None
extreme_sens = compute_extreme_sensitivity(complete, med_sample, acme_obs)
print(f"ACME observed={extreme_sens.get('acme_observed')}, "
      f"conservative_bound={extreme_sens.get('acme_weighted_conservative_assuming_zeros')}, "
      f"survives={extreme_sens.get('survives_conservative_bound')}")

# ── Sequential ignorability check ─────────────────────────────────────────
if len(med_sample) >= 10:
    seq_ign = sequential_ignorability_check(
        med_sample, outcome_col="v2jucomp", mediator_col="pse_share_period"
    )
else:
    seq_ign = {"error": "Insufficient obs", "n_obs": len(med_sample)}

print(f"rho={seq_ign.get('rho_residual_correlation')}, p={seq_ign.get('rho_p_value')}, "
      f"concern={seq_ign.get('sequential_ignorability_concern')}")

## Section 5 — Reverse Causality & Robustness Tests

1. **Gini ← lagged LDem**: does past democracy reduce inequality? Significant negative β = reverse causality concern
2. **IMF SAP → ΔSocProt**: does IMF program share predict SocProt change? Descriptive only
3. **Lagged SocProt sensitivity**: replace contemporaneous SocProt with one-period lag in DD triple

In [ ]:
def test_reverse_causality_gini_ldem(df: pd.DataFrame) -> dict:
    df = df.sort_values(["country_iso3", "period_start"]).copy()
    df["ldem_lag1"] = df.groupby("country_iso3")["v2x_libdem"].shift(1)
    df = df.dropna(subset=["gini_disp", "ldem_lag1"])
    if len(df) < 10:
        return {"error": "Insufficient obs for reverse causality test", "n_obs": len(df)}

    df["gini_w"] = df["gini_disp"] - df.groupby("country_iso3")["gini_disp"].transform("mean")
    df["ldem_w"] = df["ldem_lag1"] - df.groupby("country_iso3")["ldem_lag1"].transform("mean")
    period_dummies = pd.get_dummies(df["period_start"], prefix="pd", drop_first=True).astype(float)
    X = add_constant(pd.concat([df[["ldem_w"]].reset_index(drop=True), period_dummies.reset_index(drop=True)], axis=1))
    y = df["gini_w"].reset_index(drop=True)
    mod = OLS(y, X, missing="drop").fit(cov_type="HC3")
    coef = safe_float(mod.params.get("ldem_w"))
    p    = safe_float(mod.pvalues.get("ldem_w"))

    result = {
        "path_a_description": "Regress gini on lagged LDem (country FE + period FE)",
        "n_obs": int(mod.nobs),
        "beta_ldem_on_gini": round(coef, 6) if coef is not None else None,
        "p_value": round(p, 4) if p is not None else None,
        "reverse_causality_concern": bool(p < 0.05) if p is not None else None,
        "interpretation": (
            "Significant negative effect means democracies reduce inequality; "
            "if so, Gini is endogenous to LDem."
        ),
    }
    logger.info(f"Reverse causality Gini←LDem: β={result['beta_ldem_on_gini']}, p={result['p_value']}")
    return result


def test_imf_sap_socprot_correlation(df: pd.DataFrame) -> dict:
    df = df.sort_values(["country_iso3", "period_start"]).copy()
    df["dsocprot"] = df.groupby("country_iso3")["socprot_coverage"].diff()
    df["imf_lag"]  = df.groupby("country_iso3")["imf_sap_share"].shift(1)
    sub = df.dropna(subset=["dsocprot", "imf_lag"])
    if len(sub) < 10:
        return {"error": "Insufficient obs for IMF SAP test", "n_obs": len(sub)}

    period_dummies = pd.get_dummies(sub["period_start"], prefix="pd", drop_first=True).astype(float)
    X = add_constant(pd.concat([sub[["imf_lag"]].reset_index(drop=True), period_dummies.reset_index(drop=True)], axis=1))
    y = sub["dsocprot"].reset_index(drop=True)
    mod  = OLS(y, X, missing="drop").fit(cov_type="HC3")
    coef = safe_float(mod.params.get("imf_lag"))
    p    = safe_float(mod.pvalues.get("imf_lag"))
    corr = safe_float(sub[["dsocprot", "imf_lag"]].corr().iloc[0, 1])

    result = {
        "path_b_description": "Correlation of lagged IMF SAP share with SocProt change (descriptive, not IV)",
        "n_obs": int(mod.nobs),
        "raw_correlation": round(corr, 4) if corr is not None else None,
        "beta_imfsap_on_dsocprot": round(coef, 6) if coef is not None else None,
        "p_value": round(p, 4) if p is not None else None,
        "interpretation": (
            "Negative beta means IMF programs precede SocProt reductions. "
            "Descriptive only — not used as IV per hypothesis exclusion restriction decision."
        ),
    }
    logger.info(f"IMF SAP→SocProt: β={result['beta_imfsap_on_dsocprot']}, p={result['p_value']}")
    return result


def run_lagged_socprot_sensitivity(complete: pd.DataFrame) -> dict:
    df = complete.sort_values(["country_iso3", "period_start"]).copy()
    df["socprot_lag1"] = df.groupby("country_iso3")["socprot_coverage"].shift(1)
    df["mean_sp_lag"]  = df.groupby("country_iso3")["socprot_lag1"].transform("mean")
    df["e_socprot_lag1"] = df["socprot_lag1"] - df["mean_sp_lag"]
    sub = df.dropna(subset=["education", "gini_disp", "socprot_lag1", "v2x_libdem"])

    if len(sub) < 10:
        return {
            "error": "Insufficient obs for lagged SocProt sensitivity",
            "lagged_sensitivity_underpowered": True,
            "n_obs": len(sub),
        }

    return run_dd_triple(
        df=sub,
        e_educ_col="e_education",
        e_gini_col="e_gini_disp",
        e_mod_col="e_socprot_lag1",
        outcome_col="v2x_libdem",
        country_col="country_iso3",
        label="SocProt_lagged_t5",
    )


rev_gini = test_reverse_causality_gini_ldem(panel)
rev_imf  = test_imf_sap_socprot_correlation(panel)
lag_sens = run_lagged_socprot_sensitivity(complete)

print(f"Gini←LDem: β={rev_gini.get('beta_ldem_on_gini')}, p={rev_gini.get('p_value')}, concern={rev_gini.get('reverse_causality_concern')}")
print(f"IMF→ΔSocProt: β={rev_imf.get('beta_imfsap_on_dsocprot')}, p={rev_imf.get('p_value')}")
print(f"Lagged SocProt: β₇={lag_sens.get('beta7')}, p={lag_sens.get('p7')}, n={lag_sens.get('n_obs')}")

## Section 6 — Visualization

Four-panel summary figure:
1. **DD β₇ comparison**: SocProt vs PSE triple-interaction coefficients with error bars
2. **Marginal effects of education** at low/high moderator (25th/75th percentile)
3. **Baron-Kenny mediation path diagram** — ACME magnitudes for three outcomes
4. **PSE coverage by quadrant** — bar chart of PSE % in each Gini × SocProt quadrant

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle("PSE Moderator Comparison + Causal Mediation\n(demo subset, N=36 obs / 18 countries)", fontsize=12)

# ── Panel 1: β₇ comparison ───────────────────────────────────────────────
ax = axes[0, 0]
labels = ["SocProt\n(within-DD)", "PSE share\n(cross-sectional)"]
betas  = [result_socprot.get("beta7") or 0, result_pse.get("beta7") or 0]
ses    = [result_socprot.get("se7") or 0, result_pse.get("se7") or 0]
ps     = [result_socprot.get("p7"), result_pse.get("p7")]
colors = ["steelblue" if (p is not None and p < 0.1) else "lightgray" for p in ps]
bars = ax.bar(labels, betas, yerr=[1.96 * s for s in ses], color=colors, capsize=6, alpha=0.85)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
for bar, p in zip(bars, ps):
    label = f"p={p:.3f}" if p is not None else "p=N/A"
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(ses) * 2.5,
            label, ha="center", va="bottom", fontsize=9)
ax.set_title("Triple Interaction β₇ (Edu×Gini×Mod→LDem)")
ax.set_ylabel("β₇ coefficient")

# ── Panel 2: Marginal effects at low/high moderator ──────────────────────
ax = axes[0, 1]
me_data = {
    "SocProt": (result_socprot.get("sign_reversal", {}).get("me_at_low_moderator", 0),
                result_socprot.get("sign_reversal", {}).get("me_at_high_moderator", 0)),
    "PSE":     (result_pse.get("sign_reversal", {}).get("me_at_low_moderator", 0),
                result_pse.get("sign_reversal", {}).get("me_at_high_moderator", 0)),
}
x = np.arange(len(me_data))
w = 0.35
mod_labels = list(me_data.keys())
low_vals  = [me_data[k][0] for k in mod_labels]
high_vals = [me_data[k][1] for k in mod_labels]
ax.bar(x - w/2, low_vals,  w, label="Low mod (P25)", color="cornflowerblue", alpha=0.85)
ax.bar(x + w/2, high_vals, w, label="High mod (P75)", color="tomato", alpha=0.85)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels(mod_labels)
ax.set_title("Marginal Effect of Education\n(Gini at P75, moderator at P25/P75)")
ax.set_ylabel("∂LDem/∂Education")
ax.legend(fontsize=8)

# ── Panel 3: ACME magnitudes ──────────────────────────────────────────────
ax = axes[1, 0]
outcomes = ["Judicial\n(v2jucomp)", "NGO\n(v2cseeorgs)", "LDem\n(v2x_libdem)"]
acmes = [
    med_judicial.get("acme") or 0,
    med_ngo.get("acme") or 0,
    med_libdem.get("acme") or 0,
]
cis_low  = [
    (med_judicial.get("acme_ci_95") or [0, 0])[0],
    (med_ngo.get("acme_ci_95")      or [0, 0])[0],
    (med_libdem.get("acme_ci_95")   or [0, 0])[0],
]
cis_high = [
    (med_judicial.get("acme_ci_95") or [0, 0])[1],
    (med_ngo.get("acme_ci_95")      or [0, 0])[1],
    (med_libdem.get("acme_ci_95")   or [0, 0])[1],
]
yerr_low  = [a - l for a, l in zip(acmes, cis_low)]
yerr_high = [h - a for a, h in zip(acmes, cis_high)]
ax.bar(outcomes, acmes,
       yerr=[yerr_low, yerr_high],
       color="mediumpurple", capsize=6, alpha=0.85)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title(f"Baron-Kenny ACME\n(PSE→Outcome, {N_BOOTSTRAP} bootstrap reps)")
ax.set_ylabel("ACME (indirect effect)")

# ── Panel 4: PSE coverage by Gini × SocProt quadrant ────────────────────
ax = axes[1, 1]
quad_labels = [k.replace("highGini=", "G=").replace("_highSocProt=", " S=")
               for k in pse_missing_table.keys()]
pse_pct = [v["pse_coverage_pct"] for v in pse_missing_table.values()]
n_obs_q = [v["n_observations"] for v in pse_missing_table.values()]
bar_colors = ["#4daf4a" if p >= 50 else "#e41a1c" for p in pse_pct]
bars = ax.bar(quad_labels, pse_pct, color=bar_colors, alpha=0.85)
ax.axhline(50, color="gray", linewidth=0.8, linestyle="--", label="50% threshold")
for bar, n in zip(bars, n_obs_q):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f"n={n}", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, 110)
ax.set_title("PSE Coverage by Quadrant\n(G=Gini, S=SocProt, above/below median)")
ax.set_ylabel("PSE data coverage (%)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
print("Visualization complete.")